In [35]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.preprocessing import OneHotEncoder
import pickle


Load the dataset

In [36]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### Preprocessing the data
- [x] Gender
- [x] Geography


#### Data dropping

In [37]:
data = data.drop(['RowNumber','CustomerId','Surname'], axis = 1) #Dropping irrelevent data
data.head()


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


#### Encoding categorical variable

##### Label encoder

In [38]:
Label_encoder_gender = LabelEncoder()
data['Gender'] = Label_encoder_gender.fit_transform(data['Gender'])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


##### One hot encoding

In [39]:
ohe_geo = OneHotEncoder()
geo_encoder = ohe_geo.fit_transform(data[['Geography']])

In [40]:
ohe_geo.get_feature_names_out()

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [41]:
geo_encoded = pd.DataFrame(geo_encoder.toarray(), columns=ohe_geo.get_feature_names_out())

In [42]:
data = pd.concat([data.drop('Geography', axis = 1), geo_encoded], axis = 1)


In [43]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


#### Saving the encoders and scaler

In [44]:
with open('Label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(Label_encoder_gender,file)
with open('ohe_geo.pkl', 'wb') as file:
    pickle.dump(ohe_geo, file)

#### Divide dataset into dependant and independant feautures

In [45]:
X = data.drop('Exited', axis=1) #Independant
Y = data['Exited'] #Dependatn

### Splitting the data in training and testing sets

In [46]:
X_Train, X_Test, Y_Train, Y_test = train_test_split(X,Y, test_size=0.2, random_state=67)

### Scaling the data

In [47]:
scaler = StandardScaler()
X_Train = scaler.fit_transform(X_Train)
X_Test = scaler.transform(X_Test)

In [48]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

# ANN Implementation

In [49]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

## Building model

In [50]:
model = Sequential([

    Dense(64, activation='relu', input_shape=(X_Train.shape[1], )), #Hidden Layer 1, connected with input layer
    Dense(32, activation='relu'), #Hidden Layer 2
    Dense(1, activation='sigmoid') #Output layer

])

In [51]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                832       
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


## Model compile

In [52]:
opt = tf.keras.optimizers.Adam(learning_rate=0.025) #Variable learning rate that we can set
loss = tf.keras.losses.BinaryCrossentropy() #Setting loss function

In [53]:
model.compile(optimizer=opt, loss=loss, metrics=['accuracy']) 


You may also give direct arguments in the complie functions
```python
model.compile(optimizer="adam", loss="binary_crossentroy", metrics=['accuracy'])
```
But this way you cannot set your custom learning rate

### Setup TensorBoard

In [54]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

In [55]:
tf_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

### Setup early stopping

In [56]:
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

## Training the model

In [57]:
history = model.fit(

    X_Train, Y_Train, validation_data=(X_Test, Y_test), epochs=100,
    callbacks = [tf_callback, early_stopping_callback]
)

Epoch 1/100
250/250 [==============================] - 1s 2ms/step - loss: 0.3996 - accuracy: 0.8299 - val_loss: 0.3681 - val_accuracy: 0.8625
Epoch 2/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3649 - accuracy: 0.8485 - val_loss: 0.3628 - val_accuracy: 0.8665
Epoch 3/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3570 - accuracy: 0.8528 - val_loss: 0.3474 - val_accuracy: 0.8585
Epoch 4/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3496 - accuracy: 0.8536 - val_loss: 0.4159 - val_accuracy: 0.8470
Epoch 5/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3522 - accuracy: 0.8554 - val_loss: 0.3534 - val_accuracy: 0.8620
Epoch 6/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3476 - accuracy: 0.8569 - val_loss: 0.3545 - val_accuracy: 0.8640
Epoch 7/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3460 - accuracy: 0.8550 - val_loss: 0.3503 - val_accuracy: 0.8605

In [58]:
model.save('model.keras') #Saving the model as h5

### Load tensor board

In [59]:
%reload_ext tensorboard

In [60]:
import pkg_resources
%tensorboard --logdir logs/fit/

Reusing TensorBoard on port 6006 (pid 20680), started 9:26:45 ago. (Use '!kill 20680' to kill it.)